In [1]:
import os
import warnings

# Silenciar advertencias de Python y arranque de Spark/Hadoop
warnings.filterwarnings("ignore")
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=false pyspark-shell"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ED_03_02_Fundamentos_RDD") \
    .config("spark.driver.extraJavaOptions", "-Dlog4j.rootCategory=ERROR,console") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"✅ Spark iniciado correctamente y limpio. Versión: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 00:24:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark iniciado correctamente y limpio. Versión: 3.5.3


In [ ]:
Ejercicio 1: Word Count Básico

In [2]:
# Ejercicio 1: Word Count Básico
texto = ["hola mundo", "hola spark", "mundo spark"]
rdd_texto = sc.parallelize(texto)

conteo_palabras = (
    rdd_texto
    .flatMap(lambda linea: linea.split(" "))
    .map(lambda palabra: (palabra, 1))
    .reduceByKey(lambda a, b: a + b)
)

print("Ejercicio 1 - Conteo de palabras:")
for palabra, total in conteo_palabras.collect():
    print(f"  {palabra}: {total}")

Ejercicio 1 - Conteo de palabras:
  mundo: 2
  hola: 2
  spark: 2


In [ ]:
Ejercicio 2: Análisis de Ventas

In [3]:
# Ejercicio 2: Análisis de Ventas
ventas = [("Madrid", 100), ("Barcelona", 200), ("Madrid", 50)]
rdd_ventas = sc.parallelize(ventas)

resultado_ventas = (
    rdd_ventas
    .reduceByKey(lambda a, b: a + b)          # 1. Suma las ventas por ciudad
    .filter(lambda x: x[1] > 100)            # 2. Filtra ciudades con ventas > 100
    .sortBy(lambda x: x[1], ascending=False) # 3. Ordena de mayor a menor
)

print("Ejercicio 2 - Ciudades con ventas > 100 (mayor a menor):")
print(resultado_ventas.collect())

Ejercicio 2 - Ciudades con ventas > 100 (mayor a menor):
[('Barcelona', 200), ('Madrid', 150)]


In [ ]:
Ejercicio 3: Filtrado y Transformación

In [4]:
# Ejercicio 3: Filtrado y Transformación
precios_usd = [10, 25, 40, 5, 100, 15]
rdd_precios = sc.parallelize(precios_usd)

precios_eur = (
    rdd_precios
    .filter(lambda p: p >= 20)
    .map(lambda p: round(p * 0.9, 2))
)

print("Ejercicio 3 - Precios >= 20 USD convertidos a EUR (x0.9):")
print(precios_eur.collect())

Ejercicio 3 - Precios >= 20 USD convertidos a EUR (x0.9):
[22.5, 36.0, 90.0]


In [ ]:
Ejercicio 4: Procesamiento de CSV

In [5]:
# Ejercicio 4: Procesamiento de CSV
csv_datos = ["Leche,Huevos,Pan", "Manzanas,Peras", "Leche,Chocolate"]
rdd_csv = sc.parallelize(csv_datos)

productos_unicos = (
    rdd_csv
    .flatMap(lambda linea: linea.split(","))
    .distinct()
)

lista_unicos = productos_unicos.collect()
print("Ejercicio 4 - Productos únicos:", lista_unicos)
print("Total de productos únicos:", productos_unicos.count())

Ejercicio 4 - Productos únicos: ['Pan', 'Peras', 'Leche', 'Huevos', 'Manzanas', 'Chocolate']
Total de productos únicos: 6
